In [1]:
import os
from reasoning.tools.utils import load_model_with_vllm
from reasoning.evaluator.math_grader import math_equal, extract_answer
from datasets import load_dataset
from vllm import SamplingParams
import wandb
import time
from reasoning.tools.logger import load_config, apply_config
import numpy as np
import json
import argparse
from reasoning.inference.tree import Path
from reasoning.models.model import ValueModel_qwen




In [2]:


dataset = load_dataset('HuggingFaceH4/MATH-500')
dataset = dataset['test']
# model_name = 'Qwen/Qwen2.5-7B-Instruct'
# model_name = 'meta-llama/Llama-3.2-1B-Instruct'
model_name = "Qwen/Qwen2.5-7B-Instruct"
model, tokenizer = load_model_with_vllm(model_name, task='auto', tensor_parallel_size=4, gpu_memory_utilization=0.9)
reward_model = ValueModel_qwen(device = "auto")
tokenizer.pad_token = tokenizer.eos_token 


INFO 04-19 20:22:23 config.py:510] This model supports multiple tasks: {'embed', 'classify', 'reward', 'generate', 'score'}. Defaulting to 'generate'.
INFO 04-19 20:22:23 config.py:1310] Defaulting to use mp for distributed inference
INFO 04-19 20:22:23 llm_engine.py:234] Initializing an LLM engine (v0.6.6.post1) with config: model='Qwen/Qwen2.5-7B-Instruct', speculative_config=None, tokenizer='Qwen/Qwen2.5-7B-Instruct', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=32768, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=4, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto, quantization_param_path=None, device_config=cuda, decoding_config=DecodingConfig(guided_decoding_backend='xgrammar'), observability_config=ObservabilityConfig(otlp_traces_endpoint=None, collect_model_

Loading safetensors checkpoint shards:   0% Completed | 0/4 [00:00<?, ?it/s]


INFO 04-19 20:22:29 model_runner.py:1099] Loading model weights took 3.5546 GB
(VllmWorkerProcess pid=4081607) INFO 04-19 20:22:29 model_runner.py:1099] Loading model weights took 3.5546 GB
(VllmWorkerProcess pid=4081605) INFO 04-19 20:22:29 model_runner.py:1099] Loading model weights took 3.5546 GB
(VllmWorkerProcess pid=4081606) INFO 04-19 20:22:29 model_runner.py:1099] Loading model weights took 3.5546 GB
(VllmWorkerProcess pid=4081606) (VllmWorkerProcess pid=4081605) INFO 04-19 20:22:34 worker.py:241] Memory profiling takes 4.88 seconds
INFO 04-19 20:22:34 worker.py:241] Memory profiling takes 4.88 seconds
(VllmWorkerProcess pid=4081605) (VllmWorkerProcess pid=4081606) INFO 04-19 20:22:34 worker.py:241] the current vLLM instance can use total_gpu_memory (47.43GiB) x gpu_memory_utilization (0.90) = 42.69GiB
INFO 04-19 20:22:34 worker.py:241] the current vLLM instance can use total_gpu_memory (47.43GiB) x gpu_memory_utilization (0.90) = 42.69GiB
(VllmWorkerProcess pid=4081605) (VllmW

Capturing CUDA graph shapes:  97%|█████████▋| 34/35 [00:19<00:00,  1.77it/s]

(VllmWorkerProcess pid=4081606) INFO 04-19 20:23:02 model_runner.py:1535] Graph capturing finished in 21 secs, took 0.40 GiB
(VllmWorkerProcess pid=4081607) INFO 04-19 20:23:02 model_runner.py:1535] Graph capturing finished in 21 secs, took 0.40 GiB


Capturing CUDA graph shapes: 100%|██████████| 35/35 [00:20<00:00,  1.69it/s]

INFO 04-19 20:23:02 model_runner.py:1535] Graph capturing finished in 21 secs, took 0.40 GiB
(VllmWorkerProcess pid=4081605) INFO 04-19 20:23:02 model_runner.py:1535] Graph capturing finished in 21 secs, took 0.40 GiB
INFO 04-19 20:23:02 llm_engine.py:431] init engine (profile, create kv cache, warmup model) took 32.38 seconds


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Some weights of the model checkpoint at Qwen/Qwen2.5-Math-PRM-7B were not used when initializing Qwen2ForProcessRewardModel: ['lm_head.weight']
- This IS expected if you are initializing Qwen2ForProcessRewardModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing Qwen2ForProcessRewardModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [3]:


# inference hyperparameters
num_return_sequences = 2
max_new_tokens = 1024
temperature = 0.7
top_p = 0.9
batch_size = 10
system_prompt = """

    Solve the following math problem efficiently and clearly:
    
    - For simple problems (2 steps or fewer):
      Provide a concise solution with minimal explanation.
    
    - For complex problems (3 steps or more):
      Use this step-by-step format:
    
      ## Step 1: [Concise description]
      [Brief explanation and calculations]
    
      ## Step 2: [Concise description]
      [Brief explanation and calculations]
    
      ...
    
      Regardless of the approach, always conclude with:
    
      Therefore, the final answer is: $\boxed{answer}$. I hope it is correct.
    
      Where [answer] is just the final number or expression that solves the problem.
"""

# name of the results file
config_name =  'test'

# Also set the seed for the sampling params 
sampling_params = SamplingParams(
    temperature=temperature,
    max_tokens=max_new_tokens,
    n=num_return_sequences,
    top_p=top_p,
    stop_token_ids=[tokenizer.eos_token_id],
    skip_special_tokens = True,
    include_stop_str_in_output = False,
) # shouldn't set seed for random sampling


In [7]:

questions = ["Convert the point $(0,3)$ in rectangular coordinates to polar coordinates. Enter your answer in the form $(r,\theta),$ where $r > 0$ and $0 \le \theta < 2 \pi.$"]
conversations = [[
    {
        "role": "system",
        "content": system_prompt
    },
    {
        "role": "user",
        "content": question
    }
] for question in questions]

outputs = model.chat(conversations, sampling_params)
print(outputs[0].outputs[0].text)




<>:1: SyntaxWarning: invalid escape sequence '\l'
<>:1: SyntaxWarning: invalid escape sequence '\l'
/tmp/ipykernel_4080882/4095116080.py:1: SyntaxWarning: invalid escape sequence '\l'
  questions = ["Convert the point $(0,3)$ in rectangular coordinates to polar coordinates. Enter your answer in the form $(r,\theta),$ where $r > 0$ and $0 \le \theta < 2 \pi.$"]
Processed prompts:  50%|█████     | 1/2 [00:01<00:01,  1.93s/it, est. speed input: 107.36 toks/s, output: 172.18 toks/s]

## Step 1: Calculate r
To find $r$, we use the formula $r = \sqrt{x^2 + y^2}$. For the point $(0,3)$, we substitute $x = 0$ and $y = 3$.

## Step 2: Calculate theta
Since $x = 0$ and $y = 3$, the point lies on the positive y-axis. The angle $\theta$ for this position is $\frac{\pi}{2}$.

Therefore, the final answer is: $\boxed{(3, \frac{\pi}{2})}$.


: 

In [7]:
import importlib
import reasoning.inference.beam
beam_module = importlib.reload(reasoning.inference.beam)
BeamInference = beam_module.BeamInference


inference = BeamInference(model, tokenizer,sampling_params,config_name,reward_model,max_steps=20,beam_width=3)

start_time = time.time()
for i in range(0, len(dataset)):
    question = dataset['problem'][i]
    answer = dataset['solution'][i]            
    accuracy = inference.inference(question, answer)
    print("Accuracy: {}".format(accuracy))
inference.reset()
wandb.log({"accuracy": accuracy})
end_time = time.time()
print(f"parallel size: {num_return_sequences}")
print(f"generated tokens per sample in average: {np.mean(inference.num_generated_tokens) * num_return_sequences}")
wandb.log({"generated tokens per sample in average": np.mean(inference.num_generated_tokens) * num_return_sequences})
print("Time taken: {} seconds".format(end_time - start_time))

wandb.finish()

Processed prompts:   0%|          | 0/3 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts:  33%|███▎      | 1/3 [00:00<00:01,  1.33it/s, est. speed input: 403.68 toks/s, output: 465.15 toks/s]
We detected that you are passing `past_key_values` as a tuple and this is deprecated and will be removed in v4.43. Please use an appropriate `Cache` class (https://huggingface.co/docs/transformers/v4.41.3/en/internal/generation_utils#transformers.Cache)


standardized next step: Step 1: To convert the point $(0,3)$ from rectangular coordinates to polar coordinates, we need to find the radius $r$ and the angle $\theta$. The relationship between rectangular and polar coordinates is given by $x = r \cos \theta$ and $y = r \sin \theta$. Since the point is $(0,3)$, we can substitute $x=0$ and $y=3$ into these equations to get $0 = r \cos \theta$ and $3 = r \sin \theta$. We can then solve for $r$ and $\theta$ using these equations.

standardized next step: Step 1: To convert the point $(0,3)$ from rectangular coordinates to polar coordinates, we need to find the distance $r$ and the angle $\theta$ from the origin. The distance $r$ is found using the formula $r = \sqrt{x^2 + y^2}$, where $x$ and $y$ are the rectangular coordinates of the point.

standardized next step: Step 1: To convert the point $(0,3)$ from rectangular coordinates to polar coordinates, we can use the following steps:
1. Calculate the radius $r$ using the formula $r = \sqrt{

Processed prompts:  33%|███▎      | 1/3 [00:02<00:04,  2.36s/it, est. speed input: 181.24 toks/s, output: 283.71 toks/s]


standardized next step: Step 2: To find the value of $\theta$, we can use the equation $3 = r \sin \theta$, which is equivalent to $\sin \theta = \frac{3}{r}$. Using the Pythagorean identity $\sin^2 \theta + \cos^2 \theta = 1$, we can solve for $r$:

standardized next step: Step 2: To find the radius $r$, we can divide both sides of the equation $3 = r \sin \theta$ by $r$.

standardized next step: Step 2: Analyze the problem and identify the given information.
The problem requires converting the point (0,3) from rectangular coordinates to polar coordinates. We are given the rectangular coordinates (x, y) = (0, 3) and need to find the corresponding polar coordinates (r, θ).

##



...

Processed prompts:  33%|███▎      | 1/3 [00:00<00:00,  2.24it/s, est. speed input: 1299.77 toks/s, output: 431.73 toks/s]


standardized next step: Step 4: To find the value of $r$, we can substitute $x=0$ and $y=3$ into the equation $r = \sqrt{x^2 + y^2}$.

standardized next step: Step 4: We can use the fact that $x = 0$ implies $r = 0$, and we can find $\theta$ using the fact that $\tan \theta = \frac{y}{x}$, which in this case is $\tan \theta = \frac{3}{0}$. However, this is undefined, which means that the point $(0, 3)$ cannot be in polar coordinates.

standardized next step: Step 4: To find the value of $\theta$, we can use the equation $\theta = \tan^{-1}\left(\frac{y}{x}\right)$. Since $x=0$ and $y=3$, we have $\theta = \tan^{-1}(0)$.



Processed prompts:  33%|███▎      | 1/3 [00:00<00:00,  2.90it/s, est. speed input: 1806.56 toks/s, output: 392.70 toks/s]


standardized next step: Step 5: To find the value of $r$, we can substitute $x=0$ and $y=3$ into the equation $r = \sqrt{x^2 + y^2}$.

$r = \sqrt{0^2 + 3^2} = \sqrt{9} = 3$

next step is repeated！

standardized next step: Step 5: \boxed{\sqrt{0^2 + 3^2} = \sqrt{9} = 3}



ValueError: setting an array element with a sequence. The requested array has an inhomogeneous shape after 1 dimensions. The detected shape was (3,) + inhomogeneous part.

In [ ]:
import importlib
import reasoning.inference.majority
majority_module = importlib.reload(reasoning.inference.majority)
MajorityInference = majority_module.MajorityInference


inference = MajorityInference(model, tokenizer,sampling_params,config_name,reward_model,method='weighted_majority')

start_time = time.time()
for i in range(0, len(dataset), batch_size):
    questions = dataset['problem'][i:i+batch_size]
    answers = dataset['solution'][i:i+batch_size]            
    accuracy = inference.inference(system_prompt, questions, answers)
    print("Accuracy: {}".format(accuracy))
inference.reset()
wandb.log({"accuracy": accuracy})
end_time = time.time()
print(f"parallel size: {num_return_sequences}")
print(f"generated tokens per sample in average: {np.mean(inference.num_generated_tokens) * num_return_sequences}")
wandb.log({"generated tokens per sample in average": np.mean(inference.num_generated_tokens) * num_return_sequences})
print("Time taken: {} seconds".format(end_time - start_time))

wandb.finish()